# PFE ML — Comparaison de librairies (Phase A)

**Question à laquelle ce notebook répond :** *Avons-nous choisi la meilleure famille de modèle avec une seule exécution HGB, ou avons-nous eu de la chance ?*

Entraîne quatre librairies de gradient boosting sur le **même** jeu de données de 2 M de lignes corrigé en périodes, le **même** découpage train 2017–2023 / test 2024, et avec des **hyperparamètres par défaut comparables** (~400 itérations de boosting, taux d'apprentissage 0,05, régularisation L2 légère, gestion du déséquilibre de classes via le mécanisme natif de chaque librairie). La seule chose qui change entre les exécutions est la librairie.

## En quoi consiste cette étape, d'un point de vue méthodologique

C'est la **Phase A** d'une étude de sélection de modèle en trois phases :

| Phase | Question | Ce notebook |
|---|---|---|
| **A. Comparaison de librairies** | Quelle librairie de boosting l'emporte aux réglages par défaut ? | **Oui — celui-ci.** |
| B. Optimisation des hyperparamètres | Peut-on améliorer le top 2 par recherche aléatoire ? | Notebook suivant |
| C. Stabilité temporelle | Le gagnant tient-il la route avec 2023 en test (au lieu de 2024) ? | Notebook suivant |
| D. Interprétabilité | Sur quoi le gagnant s'appuie-t-il réellement ? (analyse SHAP) | Chapitre du mémoire |

Garder les quatre phases séparées permet d'attribuer individuellement chaque contribution.

## Limites honnêtes — à lire avant d'interpréter les résultats

- **Découpage temporel unique, pas de validation croisée.** Une librairie qui gagne ici de 0,5 pp d'AUC pourrait perdre sous TimeSeriesSplit. Traitez les écarts inférieurs à ~1 pp d'AUC comme du bruit.
- **Hyperparamètres par défaut.** Chaque librairie est alignée sur les itérations / taux d'apprentissage / profondeur, mais pas optimisée. Le gagnant de la Phase A n'est *pas forcément* le gagnant après la Phase B.
- **La gestion du déséquilibre de classes diffère.** HGB et LightGBM utilisent `class_weight='balanced'` ; XGBoost utilise `scale_pos_weight = N_neg / N_pos` ; CatBoost utilise `class_weights=[1, N_neg/N_pos]`. Mathématiquement équivalent pour la classification binaire, mais ce n'est pas littéralement la même ligne de code.
- **Reproductibilité.** Chaque librairie utilise `random_state=42`. Le boosting sur les mêmes données avec la même graine est déterministe d'une exécution à l'autre.

## Ce qui doit être présent sur la branche

Avant l'exécution, committez et poussez ceci sur `data-extraction` :
1. `app/tools/train_continuity_model.py` — expose désormais `--model-family {hgb,lightgbm,catboost,xgboost}` et un dispatcher `_build_model_pipeline()`.
2. `collabs/requirements-colab.txt` — ajoute `lightgbm`, `xgboost`, `catboost` (l'installation ajoute ~2 min à la section 2).

## 1. Environnement d'exécution et constantes

**CPU haute RAM.** Chaque librairie s'entraîne en 5–10 min ; l'importance par permutation ajoute 2–3 min par exécution. Total ~40–50 min pour les quatre librairies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

START_YEAR = 2017
END_YEAR = 2024
TRAIN_MAX_ROWS = 2_000_000
TARGET = 'continuity_risk_12m_label'

MODEL_FAMILIES = ['hgb', 'lightgbm', 'catboost', 'xgboost']

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('BRANCH       =', BRANCH)
print('TRAIN_CAP    =', TRAIN_MAX_ROWS)
print('LIBRARIES    =', MODEL_FAMILIES)

## 2. Mise à jour du code et installation des dépendances

Assurez-vous d'abord que la plomberie `--model-family` et les trois nouvelles versions de librairies épinglées sont committées et poussées sur `data-extraction`.

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
import importlib, sys

train_script_text = (Path(BACKEND_DIR) / 'app' / 'tools' / 'train_continuity_model.py').read_text(encoding='utf-8')
code_checks = {
    '--model-family CLI flag': '--model-family' in train_script_text,
    '_build_model_pipeline dispatcher': '_build_model_pipeline' in train_script_text,
    'MODEL_FAMILIES tuple': "MODEL_FAMILIES = (" in train_script_text,
    'CategoricalCaster present': 'class CategoricalCaster' in train_script_text,
    'StringCaster present': 'class StringCaster' in train_script_text,
}
print('Source-code readiness:')
for label, ok in code_checks.items():
    print(f'  {"OK " if ok else "FAIL"}  {label}')
if not all(code_checks.values()):
    raise SystemExit('Multi-library training plumbing is missing on the pulled branch. Commit + push and re-run.')

library_checks = {}
for module in ('lightgbm', 'catboost', 'xgboost'):
    try:
        mod = importlib.import_module(module)
        library_checks[module] = getattr(mod, '__version__', '?')
    except Exception as exc:
        library_checks[module] = f'IMPORT FAILED: {exc}'
print('\nLibrary install:')
for name, version in library_checks.items():
    print(f'  {name:10s}  {version}')
if any('IMPORT FAILED' in str(v) for v in library_checks.values()):
    raise SystemExit('One of the boosting libraries failed to import. Check the pip install output above.')

## 3. Vérification de la présence des données

In [ ]:
import json, duckdb

manifest_path = Path(f'{DATA_LAKE}/clean/company_identity/_manifest.json')
if not manifest_path.exists():
    raise SystemExit(f'Missing manifest at {manifest_path}. Run archive/period_fix_rebuild.ipynb first.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest.get('schema_version') == 2 and 'period' in manifest.get('grain', ''), 'Clean layer is not period-fixed.'

features_root = Path(f'{DATA_LAKE}/features/company_year_features')
labels_root = Path(f'{DATA_LAKE}/features/risk_labels')
if not list(features_root.rglob('*.parquet')) or not list(labels_root.rglob('*.parquet')):
    raise SystemExit('Feature or label parquet missing on Drive. Re-run the period-fix rebuild notebook.')

con = duckdb.connect()
counts = con.execute(f'''
    SELECT
        (SELECT count(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS feature_rows,
        (SELECT count(*) FROM read_parquet('{LABELS_GLOB}',   union_by_name=true)) AS label_rows
''').df()
print('Feature/label counts:')
print(counts.to_string(index=False))
con.close()

## 4. Entraînement des quatre librairies

Chaque librairie utilise le même jeu de 2 M de lignes, le même découpage temporel et les mêmes valeurs par défaut comparables. Le script d'entraînement ajoute automatiquement chaque exécution à `model_run_comparison.csv` ; à la fin de cette cellule, le CSV contient donc 4 nouvelles lignes (une par librairie).

**Durée totale attendue : 30–50 min.** Soyez attentif aux surprises dans les temps par librairie affichés ci-dessous — si CatBoost ou XGBoost prend 3× plus longtemps que HGB, cela mérite d'être noté dans le mémoire (le coût d'entraînement est une vraie contrainte de déploiement).

In [ ]:
import shlex, subprocess, sys, time

run_results = {}
for family in MODEL_FAMILIES:
    print('=' * 70)
    print(f'Training {family}')
    print('=' * 70)
    train_cmd = [
        sys.executable, '-u',
        '-m', 'app.tools.train_continuity_model',
        '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
        '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
        '--target', TARGET,
        '--train-start-year', str(START_YEAR),
        '--train-end-year', str(END_YEAR),
        '--max-rows', str(TRAIN_MAX_ROWS),
        '--min-rows', '1000',
        '--model-family', family,
    ]
    print(' '.join(shlex.quote(p) for p in train_cmd))
    start = time.time()
    subprocess.run(train_cmd, check=True)
    elapsed = time.time() - start
    metadata_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_metadata.json'
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    run_results[family] = {
        'run_name': metadata.get('run_name'),
        'run_dir': metadata.get('run_artifacts_dir'),
        'elapsed_seconds': elapsed,
        'metrics': metadata.get('metrics', {}),
    }
    metrics = metadata.get('metrics', {})
    print(
        f'\nDone ({elapsed:.0f}s). '
        f"AUC={metrics.get('roc_auc'):.4f}  "
        f"AP={metrics.get('average_precision'):.4f}  "
        f"F1@0.5={metrics.get('f1_at_0_5'):.4f}\n"
    )

print('All four libraries trained.')

## 5. Tableau comparatif tête-à-tête

In [ ]:
import pandas as pd

rows = []
for family in MODEL_FAMILIES:
    info = run_results[family]
    m = info['metrics']
    top_k = {item['segment']: item for item in m.get('top_k_analysis', [])}
    rows.append({
        'library': family,
        'training_seconds': round(info['elapsed_seconds']),
        'roc_auc': m.get('roc_auc'),
        'average_precision': m.get('average_precision'),
        'precision_at_0_5': m.get('precision_at_0_5'),
        'recall_at_0_5': m.get('recall_at_0_5'),
        'f1_at_0_5': m.get('f1_at_0_5'),
        'top_0.1pct_precision': top_k.get('top_0.1%', {}).get('precision'),
        'top_0.1pct_lift': top_k.get('top_0.1%', {}).get('lift'),
        'top_1pct_lift': top_k.get('top_1.0%', {}).get('lift'),
        'top_10pct_lift': top_k.get('top_10.0%', {}).get('lift'),
    })
shootout = pd.DataFrame(rows).sort_values('average_precision', ascending=False).reset_index(drop=True)
print('Library shootout (sorted by average precision):')
print(shootout.to_string(index=False))

shootout_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'library_shootout_phase_a.csv'
shootout.to_csv(shootout_csv, index=False)
print(f'\nSaved comparison CSV to: {shootout_csv}')

## 6. Graphiques côte à côte

Courbes ROC, courbes PR et précision à top-K pour les quatre librairies, une figure par type. C'est ce qui ira dans le chapitre du mémoire consacré à la sélection de modèle.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve

shootout_dir = Path(DRIVE_ROOT) / 'ml-artifacts' / 'library_shootout_phase_a'
shootout_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for family in MODEL_FAMILIES:
    metrics = run_results[family]['metrics']
    thresholds = metrics.get('threshold_analysis', [])
    top_k = metrics.get('top_k_analysis', [])
    label_auc = f"{family} (AUC={metrics.get('roc_auc'):.3f})"
    label_ap = f"{family} (AP={metrics.get('average_precision'):.3f})"

    pr_thr = [float(r['precision']) for r in thresholds]
    re_thr = [float(r['recall']) for r in thresholds]
    axes[0].plot(re_thr, pr_thr, marker='o', label=label_ap)

    fpr_seq = [1 - float(r.get('false_positive_rate', 0)) for r in thresholds]
    tpr_seq = re_thr
    fpr_actual = [float(r.get('false_positive_rate', 0)) for r in thresholds]
    axes[1].plot(fpr_actual, tpr_seq, marker='o', label=label_auc)

    fractions = [float(item['fraction']) for item in top_k]
    precisions = [float(item['precision']) for item in top_k]
    axes[2].plot(fractions, precisions, marker='o', label=family)

axes[0].set_title('Precision vs Recall (from threshold sweep)')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot([0, 1], [0, 1], '--', color='gray', linewidth=0.8)
axes[1].set_title('ROC (from threshold sweep)')
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
axes[1].grid(True, alpha=0.3); axes[1].legend()

axes[2].set_xscale('log')
axes[2].set_title('Precision at top-K (operational view)')
axes[2].set_xlabel('Top fraction flagged'); axes[2].set_ylabel('Precision')
axes[2].grid(True, alpha=0.3); axes[2].legend()

fig.suptitle(f'Library shootout — 2M rows, time-split 2024, comparable defaults')
fig.tight_layout(rect=[0, 0, 1, 0.96])
comparison_png = shootout_dir / 'pr_roc_topk_comparison.png'
fig.savefig(comparison_png, dpi=160)
plt.show()
print(f'Saved figure to: {comparison_png}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_positions = list(range(len(MODEL_FAMILIES)))
aucs = [shootout[shootout['library'] == f]['roc_auc'].iloc[0] for f in MODEL_FAMILIES]
aps = [shootout[shootout['library'] == f]['average_precision'].iloc[0] for f in MODEL_FAMILIES]
times = [run_results[f]['elapsed_seconds'] for f in MODEL_FAMILIES]

axes[0].bar(x_positions, aucs, color=['#0f766e', '#1d4ed8', '#b45309', '#7c3aed'])
axes[0].set_xticks(x_positions, MODEL_FAMILIES)
axes[0].set_ylabel('ROC AUC'); axes[0].set_title('ROC AUC by library')
axes[0].set_ylim(0.7, max(aucs) * 1.02)
for x, value in zip(x_positions, aucs):
    axes[0].text(x, value + 0.002, f'{value:.4f}', ha='center', fontsize=10)

axes[1].bar(x_positions, times, color=['#0f766e', '#1d4ed8', '#b45309', '#7c3aed'])
axes[1].set_xticks(x_positions, MODEL_FAMILIES)
axes[1].set_ylabel('Training seconds'); axes[1].set_title('Training time by library')
for x, value in zip(x_positions, times):
    axes[1].text(x, value + max(times) * 0.01, f'{value:.0f}s', ha='center', fontsize=10)

fig.tight_layout()
bar_png = shootout_dir / 'auc_and_time_by_library.png'
fig.savefig(bar_png, dpi=160)
plt.show()
print(f'Saved figure to: {bar_png}')

## 7. Affichage du résumé propre à chaque exécution

Affiche un `run_summary.md` rendu en Markdown pour chacune des quatre exécutions, afin de pouvoir parcourir les détails par librairie côte à côte.

In [ ]:
from IPython.display import Markdown, display

for family in MODEL_FAMILIES:
    run_dir = Path(run_results[family]['run_dir'])
    summary_path = run_dir / 'run_summary.md'
    print(f'\n## {family} — {run_dir.name}')
    if summary_path.exists():
        display(Markdown(summary_path.read_text(encoding='utf-8')))
    else:
        print(f'  (run_summary.md missing at {summary_path})')

## 8. Ajout à l'index global des exécutions

`model_run_comparison.csv` a été enrichi quatre fois pendant la section 4 (une fois par librairie) ; le fichier est donc déjà à jour. Cette cellule affiche simplement les dernières lignes pour confirmation, ainsi que la vue chronologique remontant jusqu'à l'exécution n°1.

In [ ]:
comparison_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.csv'
if comparison_csv.exists():
    runs_summary = pd.read_csv(comparison_csv)
    keep = [
        'run_name', 'model_family', 'rows', 'feature_count',
        'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
    ]
    keep = [c for c in keep if c in runs_summary.columns]
    print('All runs to date (chronological):')
    print(runs_summary[keep].to_string(index=False))
    print('\nLast four rows are the library shootout.')
else:
    print('No model_run_comparison.csv yet.')

## 9. Ce qu'il faut transmettre / Ce qu'il faut décider

### À joindre au mémoire
- `ml-artifacts/library_shootout_phase_a.csv` — le tableau comparatif principal.
- `ml-artifacts/library_shootout_phase_a/pr_roc_topk_comparison.png` — courbes PR / ROC / top-K côte à côte.
- `ml-artifacts/library_shootout_phase_a/auc_and_time_by_library.png` — compromis AUC vs temps d'entraînement.
- Les quatre dossiers `runs/*` les plus récents — `run_summary.md`, `feature_importances.csv` et graphiques de seuils propres à chaque librairie.
- Le `model_run_comparison.csv` mis à jour (désormais 9 + 4 = 13 lignes).

### À décider avant la Phase B
- **Sélectionner les 2 meilleures librairies** par précision moyenne (départager les égalités par le F1 @ 0,5). Ce sont les candidats pour l'optimisation des hyperparamètres.
- **Noter le compromis temps d'entraînement.** Si deux librairies sont à égalité en AP mais que l'une s'entraîne deux fois plus vite, c'est elle le choix pratique — surtout en cas de ré-entraînement régulier en production.
- **Si une librairie est à plus de 2 pp d'AP du leader**, écartez-la de la Phase B. Ne gaspillez pas le budget d'optimisation dessus.

### À formuler dans le rapport
*« Aux hyperparamètres par défaut, avec gestion native du déséquilibre de classes par chaque librairie, [GAGNANT] a atteint la précision moyenne la plus élevée (X,XX) sur le jeu de test 2024 mis de côté, dépassant le second [SECOND] de Y,Y pp. Ce résultat est cohérent avec la réputation de [GAGNANT] pour traiter [les catégorielles à haute cardinalité / les valeurs manquantes massives / etc.]. L'optimisation des hyperparamètres en Phase B est limitée à [GAGNANT] et [SECOND]. »*